<a href="https://colab.research.google.com/github/elenamaggiore/Thesis_Elena_Maggiore/blob/preprocessing/PREPROCESSING_oversampling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PREPROCESSING: oversampling

This notebook will perform oversampling on the PANDORA test set.
The plan is broken down in the following steps:
* Setting up parameters
* Choosing the model
* Creating the oversampling plan and verify its consistence
* Exectuing the oversampling plan
* Check if the process was actually successful



Libraries:

In [13]:
import gc
import json
import random
import re
import hashlib
import math
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score
from sklearn.metrics import f1_score

from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Oversampling will be applied on the final dataset:

In [14]:
BASE = Path("/content/drive/MyDrive/Thesis_Elena_Maggiore/Datasets_PANDORA")
INPUT_FILE = BASE / "final_complete_dataset.csv"
OUT_DIR = BASE / "Oversampling output" / "final"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_PATH = OUT_DIR / "author_split.csv"

AUTHOR_COL = "author"

General parameters:

In [15]:
# TASKS:
MBTI_TASKS = {"mbti_introverted": "introverted", "mbti_intuitive": "intuitive",
              "mbti_thinking": "thinking", "mbti_perceiving": "perceiving"}
B5_TASKS = {f"b5_{t}": t for t in ["openness", "conscientiousness", "extraversion",
                                   "agreeableness", "neuroticism"]}
TASKS = {**MBTI_TASKS, **B5_TASKS}

# BIG FIVE THRESHOLDS:
B5_THRESHOLDS = {
    "openness": 50,
    "conscientiousness": 50,
    "extraversion": 50,
    "agreeableness": 50,
    "neuroticism": 50,
}
B5_TIES_AS = None # What to do with record that are exactly on the threshold?

# CHUNKING PARAMETERS:
SEED = 11
CHUNK_WORDS = 150
MAX_CHUNKS_PER_AUTHOR = 10
MIN_COMMENT_WORDS = 10
MAX_COMMENT_WORDS = 200
MAX_RAW_SAMPLE = 200
VAL_SIZE = 0.30

# VERIFICATION:
LEAK_RE = re.compile(r"\b[ei][ns][tf][jp]s?\b|\b(?:[nsft][ei]){4}\b|TYPE_MENTION|COGFUNC_MENTION", re.I)

Oversampling parameters:

In [16]:
TARGET_RATIO = 0.8           # expected ratio
MAX_VARIANTS = 2             # max paraphrasis for the same chunk
OVERGENERATE = 1.2           # overgeneration to allow filtering
MAX_NEW_PER_TASK = 10_000

Model parameters:

In [17]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
LOAD_IN_4BIT = False
BATCH_SIZE = 16
MAX_NEW_TOKENS = 450
TEMPERATURE = 0.7
TOP_P = 0.9

LEN_RATIO = (0.6, 1.5)       # generated words/source words
MAX_COPY_OVERLAP = 0.70      # max overlap between source comment and its copy
MIN_SEMANTIC_SIM = 0.75      # minimum cosine similarity between source and generated words
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

Other options are listed here:

In [18]:
# MODEL_NAME = 'Qwen/Qwen3-4B-Instruct-2507'
# MODEL_NAME = 'microsoft/Phi-4-mini-instruct'
# MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'

# EMBED_MODEL = 'BAAI/bge-m3'
# EMBED_MODEL = 'Qwen/Qwen3-Embedding-0.6B'
# EMBED_MODEL = 'sentence-transformers/all-mpnet-base-v2'

Functions for extracting comments, chunking them, and building the dataset:

In [19]:
def label_cols(): # Only considers the columns related to the task, if you're running the code for a specific task
    cols = list(TASKS.values()) + ["mbti"]
    return list(dict.fromkeys(cols))


def load_authors(file): # Reads through the dataset and collects comments per author
    """Reads CSV and returns authors, labels, comments, and stats."""
    cols = label_cols()
    wanted = set(cols) | {AUTHOR_COL, "comments"}
    labels, cands, stats = {}, {}, Counter()

    for part in pd.read_csv(file, usecols=lambda c: c in wanted, chunksize=100, low_memory=False):
        missing = (set(TASKS.values()) | {AUTHOR_COL, "comments"}) - set(part.columns)
        if missing:
            raise KeyError(f"Colonne mancanti nel file: {sorted(missing)}")
        for row in part.to_dict("records"):
            aid = str(row[AUTHOR_COL])
            raw = row.get("comments")
            comments = json.loads(raw) if isinstance(raw, str) else []
            stats["author_rows"] += 1
            stats["comments_in"] += len(comments)

            good = []
            for c in comments:
                c = str(c).strip()
                n = len(c.split())
                if n < MIN_COMMENT_WORDS:
                    stats["too_short"] += 1
                elif n > MAX_COMMENT_WORDS:
                    stats["too_long"] += 1
                else:
                    good.append(c)
            if len(good) > MAX_RAW_SAMPLE:
                good = random.Random(f"{SEED}-{aid}").sample(good, MAX_RAW_SAMPLE)
            stats["comments_kept"] += len(good)

            cands.setdefault(aid, []).extend(good)
            labels.setdefault(aid, {c: row.get(c) for c in cols})
    gc.collect()
    auth = pd.DataFrame.from_dict(labels, orient="index").rename_axis("author_id").reset_index()
    return auth, cands, stats


def _chunk_row(aid, k, cur, n): # Creates a dictionary with all chunks
    return dict(chunk_id=f"{aid}_{k}", author_id=aid, text="\n".join(cur),
                comments_json=json.dumps(cur, ensure_ascii=False), n_words=n, n_comments=len(cur))


def build_chunks(cands): # Builds chunks
    rows, dropped = [], 0
    for aid, comments in cands.items():
        comments = list(dict.fromkeys(comments))
        random.Random(f"{SEED}-{aid}").shuffle(comments)
        cur, n, k = [], 0, 0
        for c in comments:
            cur.append(c)
            n += len(c.split())
            if n >= CHUNK_WORDS:
                rows.append(_chunk_row(aid, k, cur, n))
                k, cur, n = k + 1, [], 0
                if k >= MAX_CHUNKS_PER_AUTHOR:
                    break
        if k == 0 and n >= CHUNK_WORDS // 2:
            rows.append(_chunk_row(aid, 0, cur, n))
            k = 1
        dropped += int(k == 0)
    return pd.DataFrame(rows), dropped


def add_task_labels(df):
    """Creates taks columns (0/1/NaN)."""
    for task, col in MBTI_TASKS.items():
        s = pd.to_numeric(df[col], errors="coerce")
        bad = s.notna() & ~s.isin([0, 1])
        if bad.any():
            print(f"WARNING: {col} has {bad.sum()} non binary values -> NaN")
            s = s.mask(bad)
        df["y_" + task] = s.astype("float64")

    # Big Five thresholding
    for task, col in B5_TASKS.items():
        thr = B5_THRESHOLDS[col]
        s = pd.to_numeric(df[col], errors="coerce")
        y = pd.Series(np.where(s > thr, 1.0, np.where(s < thr, 0.0, np.nan)), index=s.index)
        if B5_TIES_AS is not None:
            y[s == thr] = float(B5_TIES_AS)
        df["y_" + task] = y.astype("float64")
    return df


def assign_split(chunks, path): # Splits each authors in training and validation
    """Splits each author in training and validation, saves on file."""
    if Path(path).exists():
        sp = pd.read_csv(path, dtype={"author_id": str})
    else:
        a = chunks.drop_duplicates("author_id")
        strat = None
        if "mbti" in a and a["mbti"].notna().all() and a["mbti"].value_counts().min() >= 2:
            strat = a["mbti"]
        tr, va = train_test_split(a["author_id"], test_size=VAL_SIZE, stratify=strat, random_state=SEED)
        sp = pd.concat([pd.DataFrame({"author_id": tr, "split": "train"}),
                        pd.DataFrame({"author_id": va, "split": "val"})], ignore_index=True)
        sp.to_csv(path, index=False)
        print("Split saved in", path, "| stratified on:", strat is not None)
    out = chunks.merge(sp, on="author_id", how="left")
    if out["split"].isna().any():
        print(f"WARNING: {out['split'].isna().sum()} chunk removed because author was not in split")
        out = out[out["split"].notna()]
    return out.reset_index(drop=True)


def build_dataset(file=INPUT_FILE, split_path=SPLIT_PATH): # Builds the dataset that will be used for oversampling
    auth, cands, stats = load_authors(file)
    print("Comments:", dict(stats))
    chunks, dropped = build_chunks(cands)
    print(f"Authors without chunks are discarded: {dropped:,}")
    chunks = add_task_labels(chunks.merge(auth, on="author_id", how="left"))
    return assign_split(chunks, split_path)

Functions for creating the oversampling plan and verify its consistence:

In [20]:
def generation_plan(train): # Calculates the number of samples to generate, for each task
    """For each task, labels the minority class, counts how many samples are need and how many to generate."""
    rows = []
    for task in TASKS:
        y = train["y_" + task].dropna()
        vc = y.value_counts()
        if len(vc) < 2:
            rows.append(dict(task=task, n_pos=int(vc.get(1.0, 0)), n_neg=int(vc.get(0.0, 0)),
                             minority=np.nan, ratio=np.nan, n_needed=0, n_to_generate=0))
            continue
        minority, n_min, n_maj = vc.idxmin(), vc.min(), vc.max()
        needed = int(max(0, min(TARGET_RATIO * n_maj - n_min, MAX_NEW_PER_TASK)))
        rows.append(dict(task=task, n_pos=int(vc.get(1.0, 0)), n_neg=int(vc.get(0.0, 0)),
                         minority=minority, ratio=round(n_min / n_maj, 3),
                         n_needed=needed, n_to_generate=int(np.ceil(needed * OVERGENERATE))))
    return pd.DataFrame(rows)


def _ratio(s): # Calculates the ratio
    vc = s.value_counts()
    return vc.min() / vc.max() if len(vc) > 1 else np.nan


def verify_oversampling(train, synth, val, tasks=None): # Checks if the oversampling was successful
    """
    train, val : Real dataframe
    synth      : Generated dataframe
                 chunk_id, src_chunk_id, author_id, task, text, y_*, len_ratio, overlap, sim
    """
    tasks = list(tasks or TASKS)
    ycols = [f"y_{t}" for t in tasks]

    # Split integrity
    assert not set(train.author_id) & set(val.author_id), "Authors in common between train and validation"
    assert synth.author_id.isin(train.author_id).all(), "Comments generated from authors not in train"
    assert synth.src_chunk_id.isin(train.chunk_id).all(), "Sources not in train"
    print("Okay: no leakage detected")

    # Validation balance
    rows = []
    for t in tasks:
        y = f"y_{t}"
        tr, sy = train[y].dropna(), synth.loc[synth.task == t, y].dropna()
        rows.append(dict(task=t, n_real=len(tr), n_synth=len(sy), ratio_before=_ratio(tr),
                         ratio_after=_ratio(pd.concat([tr, sy])), ratio_val=_ratio(val[y].dropna())))
    print("Balance:\n", pd.DataFrame(rows).round(3).to_string(index=False))

    # Quality check
    print("3. Metrics:\n", synth[["len_ratio", "overlap", "sim"]].describe().round(3).to_string())
    assert synth.len_ratio.between(*LEN_RATIO).all()
    assert (synth.overlap <= MAX_COPY_OVERLAP).all()
    assert (synth.sim >= MIN_SEMANTIC_SIM).all()
    assert synth.groupby(["src_chunk_id", "task"]).size().max() <= MAX_VARIANTS
    print("Quality check approved")

    # Label leakage check
    n_leak = synth.text.str.contains(LEAK_RE).sum()
    print(f"Generated: {n_leak} comments with label leakage")

    # Changes in label correlation
    d = (pd.concat([train, synth])[ycols].corr() - train[ycols].corr()).abs().max().max()
    print(f"Maximum change in label correlation: {d:.3f}")

    # AUC checks
    n = min(len(train), len(synth))
    X = pd.concat([train.text.sample(n, random_state=SEED), synth.text.sample(n, random_state=SEED)])
    y = [0] * n + [1] * n
    auc = cross_val_score(make_pipeline(TfidfVectorizer(max_features=20000, sublinear_tf=True),
                                        LogisticRegression(max_iter=1000)),
                          X, y, cv=5, scoring="roc_auc").mean()
    print(f"Real AUC vs generated AUC: {auc:.3f} (1 means that they can be distinguished)")

    # F1 without oversampling
    def fit_eval(tr_df, y, **kw):
        m = make_pipeline(TfidfVectorizer(max_features=20000, sublinear_tf=True, ngram_range=(1, 2)),
                          LogisticRegression(max_iter=1000, **kw))
        tr_df = tr_df.dropna(subset=[y])
        m.fit(tr_df.text, tr_df[y])
        v = val.dropna(subset=[y])
        return f1_score(v[y], m.predict(v.text), average="macro")

    res = []
    for t in tasks:
        y = f"y_{t}"
        tr = train.dropna(subset=[y])
        vc = tr[y].value_counts()
        minority = vc.idxmin()
        n_extra = int(max(0, TARGET_RATIO * vc.max() - vc.min()))
        rand = pd.concat([tr, tr[tr[y] == minority].sample(n_extra, replace=True, random_state=SEED)])
        res.append(dict(task=t,
                        base=fit_eval(tr, y),
                        class_weight=fit_eval(tr, y, class_weight="balanced"),
                        random_os=fit_eval(rand, y),
                        llm_os=fit_eval(pd.concat([tr, synth[synth.task == t]]), y)))
    print("7. Macro-F1 without oversampling:\n", pd.DataFrame(res).round(3).to_string(index=False))




Configurations: selecting "train" only oversamples on train set and uses validation set to assess quality, "None" oversamples on the whole dataset.

In [21]:
SOURCE_SPLIT = "train"

MAX_NEW_TOKENS_CAP = 700       # max generated number of tokens
MAX_ROUNDS = 3                 # generation and filter rounds
GEN_TAG = "qwen25_3b_v1"       # model

RAW_CACHE = OUT_DIR / f"raw_generations_{GEN_TAG}.jsonl"
SYNTH_PATH = OUT_DIR / "synthetic_chunks.csv"
REJECT_PATH = OUT_DIR / "rejected_synthetic.csv"
REPORT_PATH = OUT_DIR / "oversampling_report.csv"

PREAMBLE_RE = re.compile(
    r"^\s*(?:sure|certainly|of course|okay|ok|here(?:'s| is| are)|rewritten|paraphrased)\b[^\n]*:\s*(?:\n|$)"
    r"|\bas an ai\b|^\s*i(?:'m| am) sorry|^\s*i can(?:not|'t) (?:help|assist|rewrite)", re.I)
CJK_RE = re.compile(r"[\u3040-\u30ff\u3400-\u9fff\uac00-\ud7af]")

Prompt building:

In [22]:
SYSTEM_PROMPT = ("You are a careful text rewriter. You paraphrase informal Reddit comments "
                 "without changing their meaning, tone or voice.")
STYLE_HINTS = [
    "Change the sentence structure and the vocabulary as much as you can while keeping the meaning.",
    "Keep the casual voice of the author, but express the same ideas with different wording and a different sentence order.",
    "Do NOT make the comment sound like AI. It must be indistinguishable from human comments."
    # "..." more instructions can be added here
]

USER_TEMPLATE = """Rewrite the Reddit text below in your own words.

Rules:
- Keep the same meaning, opinions and level of detail. Do not add, remove or invent any fact.
- Keep the first-person voice, the informal register and the emotional tone of the author.
- {style_hint}
- Do not mention personality types, MBTI, cognitive functions, the Big Five or personality tests.
- Keep roughly the same length as the original.
- Output ONLY the rewritten text, with no introduction, no title, no notes and no quotation marks around it.

Text:
{text}"""

Functions:

In [23]:
# Prompt generating:
def build_messages(text, variant): # Function that builds the prompt
    hint = STYLE_HINTS[variant % len(STYLE_HINTS)]
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_TEMPLATE.format(style_hint=hint, text=text)}]

# Text Utilities:
def _md5(s): # Hash conversion
    return hashlib.md5(s.encode("utf-8")).hexdigest()


def _norm(t): # Normalisation
    return re.sub(r"\W+", " ", t.lower()).strip()


def _trigrams(t): # Trigrams
    w = re.findall(r"[a-z0-9']+", t.lower())
    return {tuple(w[i:i + 3]) for i in range(len(w) - 2)}

# Models:
def load_qwen(): #Loading the model: this might need changes based on the chosen model
    from transformers import AutoTokenizer, AutoModelForCausalLM

    if not torch.cuda.is_available():
        raise RuntimeError("No GPU: change runtime to T4 GPU.")
    print("GPU:", torch.cuda.get_device_name(0))

    tok = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    kw = dict(device_map="auto")
    if LOAD_IN_4BIT:
        from transformers import BitsAndBytesConfig
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                       bnb_4bit_compute_dtype=torch.float16)
    else:
        kw["torch_dtype"] = torch.float16
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **kw).eval()
    return tok, model


_EMBEDDER = None


def get_embedder(): # Imports the embedder
    global _EMBEDDER
    if _EMBEDDER is None:
        from sentence_transformers import SentenceTransformer
        _EMBEDDER = SentenceTransformer(EMBED_MODEL, device="cuda" if torch.cuda.is_available() else "cpu")
    return _EMBEDDER


# Generation:
def raw_cache_path():
    cfg = json.dumps([MODEL_NAME, LOAD_IN_4BIT, SYSTEM_PROMPT, USER_TEMPLATE, STYLE_HINTS,
                      TEMPERATURE, TOP_P, GEN_TAG], sort_keys=True)
    return OUT_DIR / f"raw_generations_{_md5(cfg)[:8]}.jsonl"


def load_cache(src_text):
    path = raw_cache_path()
    print("Cache file:", path.name)
    cache = {}
    if path.exists():
        with open(path, encoding="utf-8") as fh:
            for line in fh:
                try:
                    r = json.loads(line)
                except json.JSONDecodeError:
                    continue
                s = r["src"]
                if s in src_text and r["md5"] == _md5(src_text[s]):
                    cache[(s, r["v"])] = r
    print(f"Cache: {len(cache):,} available comments generated.")
    return cache


def _generate(tok, model, prompts, max_new, eos_ids):
    """Takes a batch and generates. If OOM, splits the batch and tries again."""
    try:
        enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_new, do_sample=True,
                                 temperature=TEMPERATURE, top_p=TOP_P, repetition_penalty=1.05,
                                 pad_token_id=tok.pad_token_id)
        gen = out[:, enc["input_ids"].shape[1]:]
        finished = torch.isin(gen, eos_ids).any(dim=1).tolist()   # If false, splits the batch.
        return list(zip(tok.batch_decode(gen, skip_special_tokens=True), finished))
    except torch.cuda.OutOfMemoryError:
        if len(prompts) == 1:
            raise
        gc.collect()
        torch.cuda.empty_cache()
        h = len(prompts) // 2
        return (_generate(tok, model, prompts[:h], max_new, eos_ids)
                + _generate(tok, model, prompts[h:], max_new, eos_ids))


def generate_jobs(jobs, src_text, tok, model, cache): # Creates jobs
    eos = model.generation_config.eos_token_id
    eos_ids = torch.tensor(eos if isinstance(eos, list) else [eos], device=model.device)
    torch.manual_seed(SEED)
    jobs = sorted(jobs, key=lambda j: len(src_text[j[0]]))     # puts texts of similar length in the same batch
    with open(raw_cache_path(), "a", encoding="utf-8") as fh:
        for i in tqdm(range(0, len(jobs), BATCH_SIZE), desc="Qwen"):
            batch = jobs[i:i + BATCH_SIZE]
            prompts = [tok.apply_chat_template(build_messages(src_text[s], v), tokenize=False,
                                               add_generation_prompt=True) for s, v in batch]
            max_src = max(len(tok(src_text[s]).input_ids) for s, _ in batch)
            max_new = min(MAX_NEW_TOKENS_CAP, int(max_src * LEN_RATIO[1]) + 30)
            for (s, v), (text, fin) in zip(batch, _generate(tok, model, prompts, max_new, eos_ids)):
                rec = dict(src=s, v=v, md5=_md5(src_text[s]), text=text.strip(), truncated=not fin)
                cache[(s, v)] = rec
                fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
            fh.flush()


# Filters:
def new_state(pool): # Sets new state
    return dict(judged=set(), acc=[], rej=[], seen=set(pool.text.map(_norm)), var_tris=defaultdict(list))


def judge_new(cache, state, src_text): # Judges each generated comment
    """Judges new generated batches and calculates metrics."""
    new = sorted(k for k in cache if k not in state["judged"])
    if not new:
        return
    emb = get_embedder()
    syn = [cache[k]["text"] for k in new]
    src = [src_text[k[0]] for k in new]
    sims = (emb.encode(src, batch_size=64, normalize_embeddings=True, show_progress_bar=False)
            * emb.encode(syn, batch_size=64, normalize_embeddings=True, show_progress_bar=False)).sum(1)

    for k, t, s_txt, sim in zip(new, syn, src, sims):
        state["judged"].add(k)
        ws, wt = len(s_txt.split()), len(t.split())
        len_ratio = wt / ws if ws else np.nan
        tri_s, tri_t = _trigrams(s_txt), _trigrams(t)
        overlap = len(tri_s & tri_t) / len(tri_s) if tri_s else 0.0
        nt = _norm(t)

        if cache[k]["truncated"]:
            reason = "truncated"
        elif wt < 5:
            reason = "empty"
        elif PREAMBLE_RE.search(t):
            reason = "preamble"
        elif CJK_RE.search(t):
            reason = "non_english"
        elif LEAK_RE.search(t):
            reason = "leak"
        elif not (LEN_RATIO[0] <= len_ratio <= LEN_RATIO[1]):
            reason = "length"
        elif overlap > MAX_COPY_OVERLAP:
            reason = "copy"
        elif sim < MIN_SEMANTIC_SIM:
            reason = "semantic"
        elif nt in state["seen"]:
            reason = "duplicate"
        elif any(len(tri_t & o) / max(1, len(tri_t)) > MAX_COPY_OVERLAP for o in state["var_tris"][k[0]]):
            reason = "variant_too_similar"          # The variant is too similar to another unit
        else:
            reason = None

        rec = dict(src_chunk_id=k[0], variant=k[1], text=t, len_ratio=len_ratio, overlap=overlap, sim=float(sim))
        if reason is None:
            state["seen"].add(nt)
            state["var_tris"][k[0]].append(tri_t)
            rec["rand"] = random.Random(f"{SEED}-{k[0]}-{k[1]}").random()
            state["acc"].append(rec)
        else:
            rec["reason"] = reason
            state["rej"].append(rec)


def accepted_df(state): # Dataframe for accepted generations only
    cols = ["src_chunk_id", "variant", "text", "len_ratio", "overlap", "sim", "rand"]
    return pd.DataFrame(state["acc"], columns=cols)


# Job Planning:
def candidate_pairs(minority_ids, cache, rng): # Counts pairs (chunk, variant) to test
    """Couples (chunk, variant) to try:"""
    pairs = []
    for v in range(MAX_VARIANTS):
        ids = [s for s in minority_ids if (s, v) not in cache]
        rng.shuffle(ids)
        pairs += [(s, v) for s in ids]
    return pairs


def run_oversampling(pool): # Runs the whole procedure
    """pool: real chunks. Returns dataframe of generated chunks."""
    pool = pool.reset_index(drop=True)
    plan = generation_plan(pool).set_index("task")
    print("Oversampling plan:\n", plan.to_string())

    # Removes leaky chunks
    leaky = pool.text.str.contains(LEAK_RE)
    print(f"Chunks with label leaking: {int(leaky.sum())}")
    src_pool = pool[~leaky]
    src_text = dict(zip(src_pool.chunk_id, src_pool.text))

    minority_ids, n_needed = {}, {}
    for task in TASKS:
        row = plan.loc[task]
        if row.n_needed <= 0 or pd.isna(row.minority):
            continue                                         # task is balanced
        ids = src_pool.loc[src_pool["y_" + task] == row.minority, "chunk_id"].tolist()
        minority_ids[task], n_needed[task] = ids, int(row.n_needed)
        if n_needed[task] > len(ids) * MAX_VARIANTS:
            print(f"WARNING {task}: {n_needed[task]} generations needed but in {len(ids)} chunks"
                  f"and MAX_VARIANTS={MAX_VARIANTS} the maximum number of generations is {len(ids) * MAX_VARIANTS}")

    cache = load_cache(src_text)
    state = new_state(pool)
    rng = random.Random(SEED)
    tok = model = None

    for rnd in range(1, MAX_ROUNDS + 1):
        judge_new(cache, state, src_text)
        acc = accepted_df(state)
        short = {t: max(0, n_needed[t] - int(acc.src_chunk_id.isin(minority_ids[t]).sum())) for t in n_needed}
        print(f"\nRound {rnd} | is missing for task: { {t: s for t, s in short.items() if s > 0} or 'none' }")

        jobs = set()
        for task, sh in short.items():
            if sh > 0:
                jobs.update(candidate_pairs(minority_ids[task], cache, rng)[:math.ceil(sh * OVERGENERATE)])
        if not jobs:
            break
        if tok is None:
            tok, model = load_qwen()
        print(f"Needed : {len(jobs):,} generations.")
        generate_jobs(sorted(jobs), src_text, tok, model, cache)
    judge_new(cache, state, src_text)

    if model is not None:                                    # free GPU
        del model, tok
        gc.collect()
        torch.cuda.empty_cache()

    acc = accepted_df(state)
    src_idx = pool.set_index("chunk_id")
    ycols = [c for c in pool.columns if c.startswith("y_")]
    rows = []
    for task, ids in minority_ids.items():
        cand = acc[acc.src_chunk_id.isin(ids)].sort_values(["variant", "rand"]).head(n_needed[task])
        for r in cand.itertuples():
            base = src_idx.loc[r.src_chunk_id]
            d = dict(chunk_id=f"syn_{r.src_chunk_id}_v{r.variant}_{task}", src_chunk_id=r.src_chunk_id,
                     author_id=base["author_id"], task=task, text=r.text)
            d.update({c: base[c] for c in ycols})
            d.update(len_ratio=r.len_ratio, overlap=r.overlap, sim=r.sim, variant=r.variant)
            rows.append(d)
    out_cols = ["chunk_id", "src_chunk_id", "author_id", "task", "text"] + ycols + \
               ["len_ratio", "overlap", "sim", "variant"]
    synth = pd.DataFrame(rows, columns=out_cols)

    # Results:
    rep = []
    for task in TASKS:
        y = pool["y_" + task].dropna()
        vc = y.value_counts()
        n_s = int((synth.task == task).sum())
        n_min = int(vc.min()) if len(vc) > 1 else 0
        n_maj = int(vc.max()) if len(vc) > 1 else 0
        rep.append(dict(task=task, n_min=n_min, n_maj=n_maj, n_needed=n_needed.get(task, 0), n_synth=n_s,
                        ratio_before=round(n_min / n_maj, 3) if n_maj else np.nan,
                        ratio_after=round((n_min + n_s) / n_maj, 3) if n_maj else np.nan))
    rep = pd.DataFrame(rep)
    rej = pd.DataFrame(state["rej"])
    print("\nOversampling results:\n", rep.to_string(index=False))
    if len(rej):
        print("\nDiscarded:\n", rej.reason.value_counts().to_string())
        print(f"Accepting ratio: {len(acc) / (len(acc) + len(rej)):.1%}")
        rej.to_csv(REJECT_PATH, index=False)
    rep.to_csv(REPORT_PATH, index=False)
    synth.to_csv(SYNTH_PATH, index=False)
    print(f"\nSaved: {SYNTH_PATH.name} ({len(synth):,} rows), {REPORT_PATH.name}, {REJECT_PATH.name}")
    return synth


def show_examples(synth, pool, n=3): # Prints some oversampled couples
    """Manually evaluation to add human control:"""
    if synth.empty:
        return
    src = pool.set_index("chunk_id").text
    for r in synth.sample(min(n, len(synth)), random_state=SEED).itertuples():
        print("=" * 80, f"\n[{r.task}] len_ratio={r.len_ratio:.2f} overlap={r.overlap:.2f} sim={r.sim:.2f}")
        print("SOURCE\n" + src[r.src_chunk_id][:900])
        print("GENERATED\n" + r.text[:900])


Execution:

In [24]:
if __name__ == "__main__":
    data = build_dataset()
    if SOURCE_SPLIT is None:
        print("WARNING: no validation will be performed. Please ensure you're not evaluating a model on the same dataset you've trained it. ")
        pool, val = data, None
    else:
        pool = data[data.split == SOURCE_SPLIT].reset_index(drop=True)
        val = data[data.split != SOURCE_SPLIT].reset_index(drop=True)

    synth = run_oversampling(pool)
    show_examples(synth, pool)

    if val is not None and len(synth):
        verify_oversampling(pool, synth, val)
    else:
        print("No validation was performed.")

Comments: {'author_rows': 377, 'comments_in': 895742, 'too_short': 166568, 'too_long': 32247, 'comments_kept': 60403}
Authors without chunks are discarded: 3
Oversampling plan:
                       n_pos  n_neg  minority  ratio  n_needed  n_to_generate
task                                                                        
mbti_introverted       1762    700       0.0  0.397       709            851
mbti_intuitive         2170    292       0.0  0.135      1444           1733
mbti_thinking          1521    941       0.0  0.619       275            330
mbti_perceiving        1376   1086       0.0  0.789        14             17
b5_openness            1714    616       0.0  0.359       755            906
b5_conscientiousness    928   1534       1.0  0.605       299            359
b5_extraversion         720   1692       1.0  0.426       633            760
b5_agreeableness        897   1465       1.0  0.612       275            330
b5_neuroticism         1036   1376       1.0  0.753 

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Round 2 | is missing for task: {'mbti_intuitive': 1148, 'b5_openness': 28}

Oversampling results:
                 task  n_min  n_maj  n_needed  n_synth  ratio_before  ratio_after
    mbti_introverted    700   1762       709      709         0.397        0.800
      mbti_intuitive    292   2170      1444      296         0.135        0.271
       mbti_thinking    941   1521       275      275         0.619        0.799
     mbti_perceiving   1086   1376        14       14         0.789        0.799
         b5_openness    616   1714       755      727         0.359        0.784
b5_conscientiousness    928   1534       299      299         0.605        0.800
     b5_extraversion    720   1692       633      633         0.426        0.800
    b5_agreeableness    897   1465       275      275         0.612        0.800
      b5_neuroticism   1036   1376        64       64         0.753        0.799

Discarded:
 reason
semantic       184
length          23
preamble        17
leak         

Preparing the file for the GitHub commit:

In [25]:
!pip -q install nbstripout

SRC = "/content/drive/MyDrive/Thesis_Elena_Maggiore/Code/PREPROCESSING: oversampling.ipynb"
DST = "/content/drive/MyDrive/Thesis_Elena_Maggiore/Code/GIT_READY/PREPROCESSING_oversampling.ipynb"

!mkdir -p "/content/drive/MyDrive/Thesis_Elena_Maggiore/Code/GIT_READY"
!cp "{SRC}" "{DST}"

!grep -c "displayName\|userId" "{DST}"
!grep -c '"outputs": \[\]' "{DST}"

1
0
